# Streaming Comparison

In [1]:
import pandas as pd

df = pd.read_csv('output_streaming_results.csv')
df['retry'] = df['test'].apply(lambda x: 'yes' if 'retry' in x else 'no')

summary = df.groupby(['agent', 'sdk', 'endpoint', 'retry']).agg({
    'status': ['sum', 'count', 'mean'],
    'calls': 'mean',
    'retries': 'mean'
}).round(2)

summary.columns = ['successes', 'total', 'rate', 'avg_calls', 'avg_retries']
summary['success_rate'] = summary['successes'].astype(int).astype(str) + '/' + summary['total'].astype(int).astype(str) + ' (' + (summary['rate'] * 100).round(0).astype(int).astype(str) + '%)'
summary = summary.reset_index()
summary = summary[['agent', 'sdk', 'endpoint', 'retry', 'success_rate', 'rate', 'avg_calls', 'avg_retries']]
summary = summary.sort_values('rate', ascending=False)
summary = summary.drop(columns=['rate'])

print(summary.to_markdown(index=False))

| agent    | sdk                       | endpoint             | retry   | success_rate   |   avg_calls |   avg_retries |
|:---------|:--------------------------|:---------------------|:--------|:---------------|------------:|--------------:|
| opencode | @ai-sdk/anthropic         | /v1/messages         | yes     | 22/22 (100%)   |       18.5  |          5.18 |
| opencode | @ai-sdk/openai-compatible | /v1/chat/completions | yes     | 21/22 (95%)    |       25.82 |          5.18 |
| claude   | @anthropic-ai/sdk         | /v1/messages         | no      | 17/22 (77%)    |       22.41 |          0    |
| opencode | @ai-sdk/anthropic         | /v1/messages         | no      | 10/22 (45%)    |       16    |          0    |
| codex    | reqwest/hyper             | /v1/chat/completions | yes     | 10/24 (42%)    |       23.79 |          4.75 |
| opencode | @ai-sdk/openai-compatible | /v1/chat/completions | no      | 5/22 (23%)     |       13.86 |          0    |
| codex    | reqwest/hyper      